In [0]:
import data.utilities.common as Commonlib

import pyspark.sql.functions as F
import yaml

from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU
from data.utilities.publishers import SnowflakeExternalTable
from pyspark.sql.types import TimestampType

In [0]:
CONFIG_BASE_PATH = "configuration/"

config_selection_path = f"{CONFIG_BASE_PATH}"
config_entries = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "configuration file")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}/{config_file}"

config = Commonlib.get_object_config(config_file_path)

In [0]:
# Instantiate medallion object
medallion = Medallion(config)

In [0]:
SOURCE_TABLES = {
    # silver source tables
    "ska1": MTU.get_fully_qualified_table_name("oracle", "crd", "ska1"),
    "skat": MTU.get_fully_qualified_table_name("oracle", "crd", "skat"),
}

In [0]:
# mapping from silver tables to gold corporate_gl table
corpt_col_mappings = {
    "corpt_chart_of_accts_cd": F.col("ska1.ktopl"),
    "corpt_gl_acct_nbr": F.regexp_replace(F.col("ska1.saknr"), "^0+", ""),
    "sap_corpt_gl_acct_nbr": F.col("ska1.saknr"),
    "proft_and_loss_acct_flg": F.when(F.col("ska1.gvtyp") == "X", F.lit("Y")).otherwise(F.lit("N")),
    "bal_sheet_acct_flg": F.when(F.col("ska1.xbilk") == "X", F.lit("Y")).otherwise(F.lit("N")),
    "corpt_gl_acct_dsc": F.col("skat.txt50"),
    "corpt_gl_acct_short_dsc": F.col("skat.txt20"),
    "gl_typ_cd": F.col("ska1.ktoks"),
    "del_flg": F.when(F.col("ska1.xloev") == "X", F.lit("Y")).otherwise(F.lit("N")),
}

In [0]:
try:
    # Base DF for join
    left_df = spark.read.table(SOURCE_TABLES["ska1"]).filter(F.col("ktopl") == "ABGP").alias("ska1")
    # other table to join to base
    skat_df = spark.read.table(SOURCE_TABLES["skat"]).filter(F.col("spras") == "E").alias("skat")

    # join silver table
    left_df = left_df.join(
        other=skat_df,
        on=(F.col("ska1.saknr") == F.col("skat.saknr")) & (F.col("ska1.ktopl") == F.col("skat.ktopl")),
        how="left",
    )

    # Transform and select final columns
    medallion.df = (
        left_df.withColumns(corpt_col_mappings)
        .select(*corpt_col_mappings.keys())
        .withColumn("__created_tsp", F.lit(medallion.start_datetime).cast(TimestampType()))
    )

except Exception as e:
    medallion.logger.error(f"Error processing corporate_gl source table reads/joins. Error message: {e}")
    raise

In [0]:
# write gold table
try:
    medallion.write_to_delta(load_type="overwrite", layer="gold")
except Exception as e:
    medallion.logger.error(f"Error writing corporate_gl table to gold layer. Error message: {e}")
    raise

In [0]:
# publish to external stage - snowflake
try:
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(medallion.catalog, medallion.schema, medallion.name)
        pub.publish_ext_table()

except Exception as e:
    medallion.logger.error(e)
    raise

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config["name"],
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
        config.get("comparison_check", {}).get("treat_nulls"),
    )
except KeyError as e:
    medallion.logger.warning(
        "Skipping comparison of legacy and modern because the required config is missing or invalid."
    )